In [ ]:
import copy
import os.path as osp

import numpy as np
import pandas as pd
from calisim.data_model import (
	DistributionModel,
	ParameterDataType,
	ParameterSpecification,
)
from calisim.optimisation import OptimisationMethod, OptimisationMethodModel
from calisim.statistics import MeanSquaredError
from pcse.base import ParameterProvider
from pcse.input import (
	DummySoilDataProvider,
	NASAPowerWeatherDataProvider,
	WOFOST72SiteDataProvider,
	YAMLAgroManagementReader,
	YAMLCropDataProvider,
)
from pcse.models import Wofost72_PP

In [ ]:
wdp = NASAPowerWeatherDataProvider(latitude=52, longitude=5)
print(wdp)

In [ ]:
sited = WOFOST72SiteDataProvider(WAV=50)
print(sited)

In [ ]:
soild = DummySoilDataProvider()
print(soild)

In [ ]:
cropd = YAMLCropDataProvider(fpath=osp.join("..", "data"), force_reload=True)
print(cropd)

In [ ]:
agro = YAMLAgroManagementReader(osp.join("..", "data", "AGMT_C2_2020.agro"))
print(agro)

In [ ]:
params = ParameterProvider(cropdata=cropd, sitedata=sited, soildata=soild)
wofost = Wofost72_PP(params, wdp, agro)
wofost.run_till_terminate()
observed_data = pd.DataFrame(wofost.get_output())
observed_data

In [ ]:
parameter_spec = ParameterSpecification(
	parameters=[
		DistributionModel(
			name="TDWI",
			distribution_name="uniform",
			distribution_args=[75, 700],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="SPAN",
			distribution_name="uniform",
			distribution_args=[20, 50],
			data_type=ParameterDataType.CONTINUOUS,
		),
	]
)

In [ ]:
def objective(
	parameters: dict, simulation_id: str, observed_data: np.ndarray | None
) -> float | list[float]:
	p = copy.deepcopy(params)
	for k in parameters:
		p.set_override(k, parameters[k])

	wofost = Wofost72_PP(p, wdp, agro)
	wofost.run_till_terminate()
	simulated_data = pd.DataFrame(wofost.get_output()).LAI.values

	metric = MeanSquaredError()
	discrepancy = metric.calculate(observed_data, simulated_data)
	return discrepancy

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

specification = OptimisationMethodModel(
	experiment_name="optuna_optimisation",
	parameter_spec=parameter_spec,
	observed_data=observed_data.LAI.values,
	method="tpes",
	output_labels=["Leaf Area Index Discrepancy"],
	directions=["minimize"],
	n_jobs=10,
	n_iterations=200,
	method_kwargs=dict(n_startup_trials=10),
)

calibrator = OptimisationMethod(
	calibration_func=objective, specification=specification, engine="optuna"
)

calibrator.specify().execute().analyze()